In [34]:
import numpy as np
import pandas as pd
from pathlib import Path

# All Formula 1 CSVs (Ergast / Kaggle "F1 World Championship 1950-2020" schema)
# This notebook lives in the same folder as the CSVs, so load from the current dir.
DATA_DIR = Path(".")

dfs = {p.stem: pd.read_csv(p) for p in sorted(DATA_DIR.glob("*.csv"))}

# Expose each table as its own variable: circuits, drivers, races, results, ...
globals().update(dfs)

for name, df in dfs.items():
    print(f"{name:<22} {df.shape[0]:>7} rows x {df.shape[1]:>2} cols")

Formula1results          27392 rows x 18 cols
circuits                    78 rows x  9 cols
constructor_results      12942 rows x  5 cols
constructor_standings    13708 rows x  7 cols
constructors               214 rows x  5 cols
driver_standings         35515 rows x  7 cols
drivers                    865 rows x  9 cols
lap_times               873755 rows x  6 cols
pit_stops                22383 rows x  7 cols
qualifying               11124 rows x  9 cols
races                     1171 rows x 18 cols
results                  27392 rows x 18 cols
seasons                     77 rows x  2 cols
sprint_results             546 rows x 17 cols
status                     140 rows x  2 cols


In [35]:
# Read a single file into a pandas DataFrame
df = pd.read_csv("results.csv")

# 1) How many rows and columns? -> (rows, columns)
print("Shape (rows, columns):", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Shape (rows, columns): (27392, 18)
Number of rows: 27392
Number of columns: 18


In [36]:
# 2) What columns are there?
print("Columns:")
print(list(df.columns))

Columns:
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']


In [37]:
# 3) What data type is each column? (object = text/string in pandas)
print(df.dtypes)

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number              object
grid                object
position            object
positionText        object
positionOrder        int64
points             float64
laps                 int64
time                object
milliseconds        object
fastestLap          object
rank                object
fastestLapTime      object
fastestLapSpeed     object
statusId             int64
dtype: object


In [38]:
# 4) Everything at once: rows, columns, dtypes, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27392 entries, 0 to 27391
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         27392 non-null  int64  
 1   raceId           27392 non-null  int64  
 2   driverId         27392 non-null  int64  
 3   constructorId    27392 non-null  int64  
 4   number           27392 non-null  object 
 5   grid             27392 non-null  object 
 6   position         27392 non-null  object 
 7   positionText     27392 non-null  object 
 8   positionOrder    27392 non-null  int64  
 9   points           27392 non-null  float64
 10  laps             27392 non-null  int64  
 11  time             27392 non-null  object 
 12  milliseconds     27392 non-null  object 
 13  fastestLap       27392 non-null  object 
 14  rank             27392 non-null  object 
 15  fastestLapTime   27392 non-null  object 
 16  fastestLapSpeed  27392 non-null  object 
 17  statusId    

In [39]:
# 5) Peek at the first 5 rows to see what the data actually looks like
df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.3,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


## 6) Merging tables — show driver *names* and *country*, not just `driverId`

`results` stores a numeric `driverId` instead of a name (to avoid repeating
"Lewis Hamilton" thousands of times). The `drivers` table is the lookup that
maps each `driverId` -> `forename`, `surname`, `nationality`, etc.

**Merging** = joining two tables on a shared key column. Here the key is
`driverId`, which exists in both tables.

- `on="driverId"` — the shared column to match rows by
- `how="left"`   — keep every row from the left table (`results`) and attach
  driver info where it matches (so we never lose a race result)

**Where does the country come from?** The `drivers` table already has a
`nationality` column (all 865 drivers filled in), so no external lookup is
needed. `nationality` is a demonym ("British"), so we map it to a real country
name ("United Kingdom") to create a clean `country` column.

In [40]:
# Grab the columns we need from the drivers lookup table.
# The drivers table ALREADY has a "nationality" column, so we don't need to
# search anywhere external for which country a driver is from.
driver_info = drivers[["driverId", "forename", "surname", "nationality"]]

# Join driver info onto every result row, matching on driverId
merged = results.merge(driver_info, on="driverId", how="left")

# Combine forename + surname into a single readable "driver" column
merged["driver"] = merged["forename"] + " " + merged["surname"]

# "nationality" is a demonym (British, Dutch, ...). Map it to an actual country
# name so we have a clean "country" column.
demonym_to_country = {
    "American": "United States", "American-Italian": "United States",
    "Argentine": "Argentina", "Argentine-Italian": "Argentina", "Argentinian": "Argentina",
    "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium",
    "Brazilian": "Brazil", "British": "United Kingdom", "Canadian": "Canada",
    "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France",
    "German": "Germany", "Hungarian": "Hungary", "Indian": "India",
    "Indonesian": "Indonesia", "Irish": "Ireland", "Italian": "Italy",
    "Japanese": "Japan", "Liechtensteiner": "Liechtenstein", "Malaysian": "Malaysia",
    "Mexican": "Mexico", "Monegasque": "Monaco", "New Zealander": "New Zealand",
    "Polish": "Poland", "Portuguese": "Portugal", "Rhodesian": "Zimbabwe",
    "Russian": "Russia", "South African": "South Africa", "Spanish": "Spain",
    "Swedish": "Sweden", "Swiss": "Switzerland", "Thai": "Thailand",
    "Uruguayan": "Uruguay", "Venezuelan": "Venezuela",
}
# .str.strip() handles a stray trailing space in the data ("Argentinian ")
merged["country"] = merged["nationality"].str.strip().map(demonym_to_country)

# Show results with the driver's name, nationality and country next to the IDs.
# Reminder: column selection uses TWO bracket pairs -> merged[[ "a", "b" ]]
merged[["driverId", "driver", "nationality", "country",
        "constructorId", "grid", "position", "points"]].head()

,driverId,driver,nationality,country,constructorId,grid,position,points
0,1,Lewis Hamilton,British,United Kingdom,1,1,1,10.0
1,2,Nick Heidfeld,German,Germany,2,5,2,8.0
2,3,Nico Rosberg,German,Germany,3,7,3,6.0
3,4,Fernando Alonso,Spanish,Spain,4,11,4,5.0
4,5,Heikki Kovalainen,Finnish,Finland,1,3,5,4.0


In [41]:
# Same pattern, chained: each extra table is just another .merge() on its ID
out = (
    results
    .merge(drivers[["driverId", "forename", "surname"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]], on="constructorId", how="left")
    .merge(races[["raceId", "name", "year"]], on="raceId", how="left",
           suffixes=("_constructor", "_race"))  # both tables have a "name" column
)

out["driver"] = out["forename"] + " " + out["surname"]
out = out.rename(columns={"name_constructor": "team", "name_race": "grand_prix"})

out[["year", "grand_prix", "driver", "team", "grid", "position", "points"]].head()

,year,grand_prix,driver,team,grid,position,points
0,2008,Australian Grand Prix,Lewis Hamilton,McLaren,1,1,10.0
1,2008,Australian Grand Prix,Nick Heidfeld,BMW Sauber,5,2,8.0
2,2008,Australian Grand Prix,Nico Rosberg,Williams,7,3,6.0
3,2008,Australian Grand Prix,Fernando Alonso,Renault,11,4,5.0
4,2008,Australian Grand Prix,Heikki Kovalainen,McLaren,3,5,4.0


## 7) `Formula1results` — `results`, but with names instead of IDs

A clean copy of the `results` table where every **ID** foreign-key column is
replaced by the real value it points to:

| results column | becomes | from lookup table |
|---|---|---|
| `raceId`        | `race` (Grand Prix name) | `races` |
| `driverId`      | `driver` (forename + surname) | `drivers` |
| `constructorId` | `constructor` (team name) | `constructors` |
| `statusId`      | `status` (e.g. "Finished") | `status` |

`resultId` is kept — it's the table's own primary key (the unique id of each
result row), not a reference to another table, so there's no name to swap in.
`number` is the driver's car number, not a foreign key, so it stays too.

In [42]:
# Build small lookup tables, renaming each "name" column so they don't collide
races_lk        = races[["raceId", "name"]].rename(columns={"name": "race"})
constructors_lk = constructors[["constructorId", "name"]].rename(columns={"name": "constructor"})
status_lk       = status[["statusId", "status"]]

# Join every lookup onto results, then build the driver's full name
f1 = (
    results
    .merge(races_lk,                                  on="raceId",        how="left")
    .merge(drivers[["driverId", "forename", "surname"]], on="driverId",   how="left")
    .merge(constructors_lk,                           on="constructorId", how="left")
    .merge(status_lk,                                 on="statusId",      how="left")
)
f1["driver"] = f1["forename"] + " " + f1["surname"]

# Same column order as `results`, with each ID column swapped for its real value
Formula1results = f1[[
    "resultId", "race", "driver", "constructor", "number", "grid",
    "position", "positionText", "positionOrder", "points", "laps", "time",
    "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed",
    "status",
]]

# Save it next to the other CSVs so it's a real dataset, then preview
Formula1results.to_csv("Formula1results.csv", index=False)
print("Formula1results:", Formula1results.shape, "-> saved to Formula1results.csv")
Formula1results.head()

Formula1results: (27392, 18) -> saved to Formula1results.csv


,resultId,race,driver,constructor,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,status
0,1,Australian Grand Prix,Lewis Hamilton,McLaren,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.3,Finished
1,2,Australian Grand Prix,Nick Heidfeld,BMW Sauber,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,Finished
2,3,Australian Grand Prix,Nico Rosberg,Williams,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,Finished
3,4,Australian Grand Prix,Fernando Alonso,Renault,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,Finished
4,5,Australian Grand Prix,Heikki Kovalainen,McLaren,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,Finished


## 8) Quick questions — most wins, podiums, and points

All three use the readable `Formula1results` table (driver names instead of IDs).

- **`position`** is text and stores `\N` for cars that didn't finish, so we compare
  it as a string: a win is `position == "1"`, a podium is `position` in 1/2/3.
- **Points** are summed per driver with `groupby`.

In [49]:
# Q1: Which driver has the most wins? (finished in position 1)
wins = Formula1results[Formula1results["position"] == "1"]["driver"].value_counts()
print("Most wins:", wins.index[0], "-", int(wins.iloc[0]))
wins.head(35)

Most wins: Lewis Hamilton - 106


driver
Lewis Hamilton        106
Michael Schumacher     91
Max Verstappen         71
Sebastian Vettel       53
Alain Prost            51
Ayrton Senna           41
Fernando Alonso        32
Nigel Mansell          31
Jackie Stewart         27
Jim Clark              25
Niki Lauda             25
Juan Fangio            24
Nico Rosberg           23
Nelson Piquet          23
Damon Hill             22
Kimi Räikkönen         21
Mika Häkkinen          20
Stirling Moss          16
Jenson Button          15
Emerson Fittipaldi     14
Jack Brabham           14
Graham Hill            14
Alberto Ascari         13
David Coulthard        13
Carlos Reutemann       12
Alan Jones             12
Mario Andretti         12
Rubens Barrichello     11
Lando Norris           11
Jacques Villeneuve     11
Felipe Massa           11
James Hunt             10
Gerhard Berger         10
Ronnie Peterson        10
Valtteri Bottas        10
Name: count, dtype: int64

In [45]:
# Q2: Which driver has the most podiums? (finished 1st, 2nd or 3rd)
podiums = Formula1results[Formula1results["position"].isin(["1", "2", "3"])]["driver"].value_counts()
print("Most podiums:", podiums.index[0], "-", int(podiums.iloc[0]))
podiums.head(10)

Most podiums: Lewis Hamilton - 206


driver
Lewis Hamilton        206
Michael Schumacher    155
Max Verstappen        128
Sebastian Vettel      122
Alain Prost           106
Fernando Alonso       106
Kimi Räikkönen        103
Ayrton Senna           80
Rubens Barrichello     68
Valtteri Bottas        67
Name: count, dtype: int64

In [46]:
# Q3: Who has the most total points? (summed across every race)
points = Formula1results.groupby("driver")["points"].sum().sort_values(ascending=False)
print("Most points:", points.index[0], "-", points.iloc[0])
points.head(10)

Most points: Lewis Hamilton - 5057.5


driver
Lewis Hamilton        5057.5
Max Verstappen        3350.5
Sebastian Vettel      3098.0
Fernando Alonso       2381.0
Kimi Räikkönen        1873.0
Valtteri Bottas       1788.0
Charles Leclerc       1650.0
Nico Rosberg          1594.5
Sergio Pérez          1585.0
Michael Schumacher    1566.0
Name: points, dtype: float64

## 9) Deeper questions — averages, countries, fastest laps, correlation, laps

A few columns are stored as **text** with `\N` for missing values, so we first
make clean numeric helper columns:

- `grid_n`   — starting position (`grid == 0` means a pit-lane start, treated as missing)
- `finish_n` — finishing position (missing = did not finish / not classified)
- `rank_n`   — fastest-lap rank (`rank_n == 1` means that driver set the race's fastest lap)

"Best average" uses a **minimum number of entries** so a driver with one lucky
race doesn't top the list.

In [ ]:
# --- Setup: numeric helper columns + a country column on Formula1results ---
fr = Formula1results.copy()
fr["grid_n"]   = pd.to_numeric(fr["grid"], errors="coerce").replace(0, np.nan)
fr["finish_n"] = pd.to_numeric(fr["position"], errors="coerce")
fr["rank_n"]   = pd.to_numeric(fr["rank"], errors="coerce")

# attach country (reuse the demonym_to_country map from section 6)
dn = drivers.copy()
dn["driver"]  = dn["forename"] + " " + dn["surname"]
dn["country"] = dn["nationality"].str.strip().map(demonym_to_country)
fr = fr.merge(dn[["driver", "country"]], on="driver", how="left")

print("helper columns ready:", [c for c in fr.columns if c.endswith("_n")] + ["country"])

In [ ]:
# Q1: Best average start & finish position (lower number = better).
# Only drivers with >= 50 entries, so the averages are meaningful.
MIN_ENTRIES = 50
g = fr.groupby("driver")
avg = pd.DataFrame({
    "entries":    g.size(),
    "avg_start":  g["grid_n"].mean().round(2),
    "avg_finish": g["finish_n"].mean().round(2),
})
avg = avg[avg["entries"] >= MIN_ENTRIES]

print("Best AVERAGE START position:")
print(avg.sort_values("avg_start").head(10)[["entries", "avg_start"]], "\n")
print("Best AVERAGE FINISH position:")
print(avg.sort_values("avg_finish").head(10)[["entries", "avg_finish"]])

In [ ]:
# Q2: Which country has the best / worst drivers?
# Metric = average points scored per race entry (only countries with >= 200 entries).
# NOTE: F1's points system changed a lot over the decades (more points awarded in
# modern eras), so this leans toward countries with strong *recent* drivers.
c = fr.groupby("country")
country = pd.DataFrame({
    "entries":    c.size(),
    "avg_points": c["points"].mean().round(3),
    "wins":       fr[fr["position"] == "1"].groupby("country").size(),
}).fillna({"wins": 0})
country = country[country["entries"] >= 200].sort_values("avg_points", ascending=False)

print("BEST countries (by avg points per entry):")
print(country.head(5), "\n")
print("WORST countries:")
print(country.tail(5))

In [ ]:
# Q3: Who has set the most fastest laps? (fastest lap of a race == rank_n 1)
fastest_laps = fr[fr["rank_n"] == 1]["driver"].value_counts()
print("Most fastest laps:", fastest_laps.index[0], "-", int(fastest_laps.iloc[0]))
fastest_laps.head(10)

In [ ]:
# Q4: Do drivers who start higher usually finish higher?
# Correlate starting position (grid_n) with finishing position (finish_n).
# Both use "1 = best", so a POSITIVE correlation means start higher -> finish higher.
pair = fr[["grid_n", "finish_n"]].dropna()
corr = pair["grid_n"].corr(pair["finish_n"])
print(f"Correlation(start, finish) = {corr:.3f}")
print("Yes — there's a clear positive link: starting near the front usually means"
      " finishing near the front (1.0 = perfect, 0 = no relationship).")

In [ ]:
# Q5: Who completed the most laps overall? (sum of laps across every race)
total_laps = fr.groupby("driver")["laps"].sum().sort_values(ascending=False)
print("Most laps completed:", total_laps.index[0], "-", int(total_laps.iloc[0]))
total_laps.head(10)